# Lab 10 Extension: Neural Networks 🧠

*Elements of Data Science: Machine Learning and Chemistry*

In Lab 10 you predicted boiling points with **k-nearest neighbors** and compared it with your **regression line** from Lab 8. This extension adds a third kind of model, a small **neural network**: the idea behind modern AI tools like ChatGPT and Claude, shrunk down to a size you can see inside.

> 📌 **This is an optional extension.** If your instructor offers extra credit for it, be sure to follow the submission steps at the end.

### What you will do
|Part|What you do|
|---|---|
|1|Learn what a single **neuron** computes|
|2|**Where neural networks shine:** a curved trend your regression line can't follow|
|3|**Back to the Lab 10 alcohols:** does a neural network beat the line?|
|4|Write a short **reflection**|
|5|Take the **quiz**|
|6|Submit (if your instructor offers extra credit)|

Cells marked ✏️ **Your turn** need you to replace the `...`. Run all cells in order from top to bottom.

## ⚙️ Initialization — DO NOT EDIT

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "shared-public")))
sys.path.insert(0, "/home/jovyan/shared-public")
from init import *
from microproject_tools import submit_plot

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

# New today: scikit-learn ("sklearn"), a popular machine learning library
try:
    from sklearn.neural_network import MLPRegressor
except ImportError:
    get_ipython().run_line_magic("pip", "install -q scikit-learn")
    from sklearn.neural_network import MLPRegressor

import warnings
warnings.filterwarnings("ignore")   # hide routine "not fully converged" messages
print("Ready ✓")

### 🧰 Your regression toolbox from Lab 8
Run this cell. We will use these four functions to fit the regression lines in this extension.

In [ ]:
def standard_units(any_numbers):
    "Convert an array of numbers to standard units."
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    "The correlation coefficient r between two columns of Table t."
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    "Slope of the regression line, in original units."
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    "Intercept of the regression line, in original units."
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))

def rmse(actual, predicted):
    "Root mean squared error, same units as the target."
    return np.sqrt(np.mean((actual - predicted) ** 2))

## 1. One Neuron at a Time

A neural network is built from many copies of one simple unit, the **neuron**. Each neuron does three things:

1. **Weigh** each input: multiply it by a **weight** $w$. A positive weight pushes the output *up*; a negative weight pushes it *down*.
2. **Add** them up, plus a **bias** $b$ (a starting offset):  $z = w_1 x_1 + w_2 x_2 + \dots + b$
3. **Activate:** pass $z$ through an **activation function** $f$:  $\text{output} = f(z)$

Steps 1 and 2 are a regression line! **Step 3 is what makes a neural network different.** It lets each neuron *bend*.

> 🎤 **Analogy:** each neuron is a judge on a talent show. The weights are what the judge pays attention to, the bias is their mood that day, and the activation is their personal rule for turning an impression into a score.

Here is one neuron computed by hand for 1-butanol (4 carbons, primary). The weights are made up, just to show the arithmetic:

In [ ]:
carbons = 4
degree = 1

w_carbons = 0.8      # made-up weights
w_degree = -0.5
b = -1.0             # made-up bias

z = w_carbons * carbons + w_degree * degree + b
print("Step 1+2: z =", np.round(z, 2))
print("Step 3:   neuron output = tanh(z) =", np.round(np.tanh(z), 3))

### Activation functions
`MLPRegressor` lets you choose one of four activations, written as strings:

| Name in code | Rule | Shape |
|---|---|---|
| `'identity'` | output = $z$ | a straight line: **no bending at all** |
| `'logistic'` | output = $1/(1+e^{-z})$ | a smooth on/off switch from 0 to 1 |
| `'tanh'` | output = $\tanh(z)$ | a smooth switch from −1 to 1 |
| `'relu'` | output = $\max(0, z)$ | off for negative $z$, a straight line for positive $z$ |

In [ ]:
z = np.arange(-4, 4.01, 0.05)
shapes = make_array("identity", "logistic", "tanh", "relu")
outputs = [z, 1 / (1 + np.exp(-z)), np.tanh(z), np.maximum(0, z)]

fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for i in np.arange(4):
    axes[i].plot(z, outputs[i], lw=3)
    axes[i].set_title(shapes.item(i))
    axes[i].set_xlabel("z (weighted sum)")
    axes[i].set_ylim(-1.5, 4)
axes[0].set_ylabel("neuron output")
plt.tight_layout()
plt.show()

### Why bother with a curved activation?

Look at `identity`: it is a straight line. If every neuron uses `identity`, then stacking neurons just gives you *weighted sums of weighted sums*, which is still one big weighted sum. **That's a regression line again, no matter how many layers you add.** You will test this claim yourself in Part 2.

The curved activations (`logistic`, `tanh`, `relu`) let each neuron **bend**. Adding many bent pieces together lets the network draw curves that a straight line can't.

### Layers

Neurons are arranged in **layers**:

```
  INPUT LAYER           HIDDEN LAYER(S)              OUTPUT LAYER
  (the features)        (neurons we choose)          (1 number)

   MW        ──┐
   carbons   ──┤        ○ ○ ○ ○ ○ ○ ○ ○
   degree    ──┼──▶     each neuron sees      ──▶      boiling point
   aliphatic ──┤        ALL the inputs
   aryl pos  ──┤
   cyclic    ──┘
```

- Every input connects to every hidden neuron, and **each connection has its own weight**.
- The network **learns** by adjusting all the weights and biases to make its predictions match the training data. This is the same "minimize the error" idea as fitting a regression line, just with many more knobs.
- "Hidden" just means these neurons are in the middle. We never see their outputs directly.

### 🧬 Sidebar: how much like a brain is this?

Artificial neurons were **inspired by real ones**. In 1943, neuroscientist Warren McCulloch and logician Walter Pitts proposed modeling a brain cell as a unit that adds up incoming signals and "fires" when the total crosses a threshold [1]. The parallels are still built into every neural network:

| Biological neuron | Artificial neuron |
|---|---|
| **Dendrites** receive signals from other neurons | **Inputs** $x_1, x_2, \dots$ |
| **Synapse strength**: some connections matter more | **Weights** $w_1, w_2, \dots$ |
| **Firing threshold** | **Bias** $b$ |
| The cell **fires an action potential** (or doesn't) | The **activation function** decides how strongly it "fires" |
| **Learning** strengthens or weakens synapses | **Training** adjusts the weights |

But the analogy has limits. Real neurons communicate with timed electrical spikes and dozens of chemical messengers. A 2021 study in *Neuron* found that copying the input–output behavior of **a single** cortical neuron took an artificial network **five to eight layers deep** [2]. One brain cell is closer to a whole network than to one of our neurons! Ideas still flow in both directions between neuroscience and AI [3].

<small>

[1] W. S. McCulloch & W. Pitts, "A logical calculus of the ideas immanent in nervous activity," *Bulletin of Mathematical Biophysics* **5**, 115–133 (1943). [doi:10.1007/BF02478259](https://doi.org/10.1007/BF02478259)<br>
[2] D. Beniaguev, I. Segev & M. London, "Single cortical neurons as deep artificial neural networks," *Neuron* **109**, 2727–2739 (2021). [doi:10.1016/j.neuron.2021.07.002](https://doi.org/10.1016/j.neuron.2021.07.002)<br>
[3] D. Hassabis, D. Kumaran, C. Summerfield & M. Botvinick, "Neuroscience-inspired artificial intelligence," *Neuron* **95**, 245–258 (2017). [doi:10.1016/j.neuron.2017.06.011](https://doi.org/10.1016/j.neuron.2017.06.011)

</small>

## 2. Where Neural Networks Shine: A Curved Trend

The boiling points of the **straight-chain alkanes** (methane CH₄, ethane C₂H₆, … eicosane C₂₀H₄₂) rise with every added carbon, but by **less each time**. Data: standard literature values (CRC Handbook), in kelvin.

#### Make the data Table

In [ ]:
alkanes = Table().with_columns(
    "name", make_array("methane", "ethane", "propane", "butane", "pentane", "hexane", "heptane",
                       "octane", "nonane", "decane", "undecane", "dodecane", "tridecane",
                       "tetradecane", "pentadecane", "hexadecane", "heptadecane", "octadecane",
                       "nonadecane", "eicosane"),
    "carbons", np.arange(1, 21),
    "bp", make_array(111.7, 184.6, 231.1, 272.7, 309.2, 341.9, 371.6, 398.8, 424.0, 447.3,
                     469.1, 489.5, 508.6, 526.7, 543.8, 560.0, 575.2, 589.5, 603.1, 616.9))
alkanes.show(5)

#### Split into training and test sets
Just like Lab 10: 15 alkanes for training, 5 hidden for testing.

In [ ]:
np.random.seed(1013)
train_alk, test_alk = alkanes.split(15)
print(train_alk.num_rows, "training and", test_alk.num_rows, "test alkanes")
print("Test alkanes have", test_alk.column("carbons"), "carbons")

### 2.1 The regression line (your toolbox)

#### Fit the line to the training set

In [ ]:
alk_slope = slope(train_alk, "carbons", "bp")
alk_intercept = intercept(train_alk, "carbons", "bp")
print(f"bp ≈ {alk_slope:.1f} × carbons + {alk_intercept:.1f}")

#### Score the line on the test set

In [ ]:
alk_line_predictions = alk_slope * test_alk.column("carbons") + alk_intercept
alk_line_rmse = rmse(test_alk.column("bp"), alk_line_predictions)
print(f"Regression line test RMSE: {alk_line_rmse:.1f} K")

#### Plot the line and its errors

In [ ]:
errors = train_alk.column("bp") - (alk_slope * train_alk.column("carbons") + alk_intercept)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(train_alk.column("carbons"), train_alk.column("bp"), label="training")
axes[0].scatter(test_alk.column("carbons"), test_alk.column("bp"), marker="^", s=70, label="test")
axes[0].plot(make_array(1, 20), alk_slope * make_array(1, 20) + alk_intercept, color="black", label="regression line")
axes[0].set_xlabel("number of carbons"); axes[0].set_ylabel("bp [K]"); axes[0].legend()
axes[1].scatter(train_alk.column("carbons"), errors)
axes[1].axhline(0, color="black")
axes[1].set_xlabel("number of carbons"); axes[1].set_ylabel("error [K]"); axes[1].set_title("Errors of the line")
plt.show()

The error plot is an upside-down **U**, just like the "twist" in the Mauna Loa microproject. A straight line is the wrong *shape* for this data, no matter how carefully we fit it.

### 2.2 A neural network with 4 neurons

#### Define a helper: standard units using the training data
We convert to standard units just like your `standard_units` function, except we always use the **training set's** mean and SD, even for test molecules, so the test set stays truly unseen.

In [ ]:
def su_using(values, reference):
    "Standard units of values, using the mean and SD of reference (the training data)."
    return (values - np.mean(reference)) / np.std(reference)

#### Prepare the training inputs and targets
sklearn wants the inputs as a **column**, not a row. `.reshape(-1, 1)` turns an array of numbers into a single column (one row per molecule).

In [ ]:
train_x = train_alk.column("carbons")
train_y = train_alk.column("bp")

X_alk = su_using(train_x, train_x).reshape(-1, 1)     # inputs: a column
y_alk = su_using(train_y, train_y)                    # target: standard units
print("X_alk shape (rows, columns):", X_alk.shape)

#### Create the neural network
Here's every argument in the line that creates it:

| Argument | Meaning |
|---|---|
| `MLPRegressor` | **M**ulti-**L**ayer **P**erceptron (old name for a neural network) that predicts a number (a *regressor*) |
| `hidden_layer_sizes=(4,)` | a **tuple** listing the neurons in each hidden layer. `(4,)` = one layer of 4 neurons; `(8, 4)` = two layers. The comma matters: `(4)` is just the number 4! |
| `activation="tanh"` | the activation used by every hidden neuron |
| `solver="lbfgs"` | the method that adjusts the weights; good for small datasets |
| `alpha=0.001` | a small penalty on big weights (bigger alpha = simpler, smoother network) |
| `max_iter=20000` | the most adjustment steps it may take |
| `random_state=0` | networks **start from random weights**; this seed makes the result repeatable |

Every sklearn model follows the same pattern: **create → `.fit(X, y)` → `.predict(X)`**.

In [ ]:
nn_alk = MLPRegressor(hidden_layer_sizes=(4,), activation="tanh",
                      solver="lbfgs", alpha=0.001, max_iter=20000, random_state=0)
nn_alk

#### Train it
This one line is where the learning happens: `.fit` adjusts all the weights and biases to match the training data.

In [ ]:
nn_alk.fit(X_alk, y_alk)
print("Training finished after", nn_alk.n_iter_, "adjustment steps")

#### Define a function to predict boiling points in kelvin
The network works in standard units, so we convert the carbons in, then convert the prediction back to kelvin.

In [ ]:
def predict_alkane_bp(model, carbons):
    "Predicted bp in kelvin for an array of carbon numbers."
    X = su_using(carbons, train_x).reshape(-1, 1)
    return model.predict(X) * np.std(train_y) + np.mean(train_y)    # undo standard units

#### Score the network on the test set

In [ ]:
nn_alk_rmse = rmse(test_alk.column("bp"), predict_alkane_bp(nn_alk, test_alk.column("carbons")))
print(f"Regression line test RMSE: {alk_line_rmse:.1f} K")
print(f"Neural network test RMSE:  {nn_alk_rmse:.1f} K")

#### Plot: the line vs. the network

In [ ]:
n_smooth = np.arange(1, 20.01, 0.1)      # many carbon values, to draw smooth curves

plt.scatter(train_alk.column("carbons"), train_alk.column("bp"), label="training")
plt.scatter(test_alk.column("carbons"), test_alk.column("bp"), marker="^", s=70, label="test")
plt.plot(n_smooth, alk_slope * n_smooth + alk_intercept, "k--", label="regression line")
plt.plot(n_smooth, predict_alkane_bp(nn_alk, n_smooth), lw=3, label="neural network (4 tanh neurons)")
plt.xlabel("number of carbons"); plt.ylabel("bp [K]"); plt.legend()
plt.show()

### 2.3 Look inside: how 4 neurons make a curve

After `.fit`, the network stores what it learned:
- `nn_alk.coefs_`: the **weights**, one array per layer
- `nn_alk.intercepts_`: the **biases**, one array per layer

(The trailing `_` is an sklearn habit meaning *"learned during fit."*)

#### Pull out the learned weights and biases

In [ ]:
w_in = nn_alk.coefs_[0][0]          # 4 weights: input -> each hidden neuron
b_in = nn_alk.intercepts_[0]        # 4 biases, one per hidden neuron
w_out = nn_alk.coefs_[1][:, 0]      # 4 weights: each hidden neuron -> output
b_out = nn_alk.intercepts_[1][0]    # 1 output bias

Table().with_columns("neuron", np.arange(1, 5), "input weight", np.round(w_in, 2),
                     "bias", np.round(b_in, 2), "output weight", np.round(w_out, 2))

#### Plot each neuron's "piece"
Each hidden neuron computes `tanh(input weight × x + bias)`, one smooth S-shaped "piece." The output adds the pieces together, each multiplied by its output weight. Each thin line is **one neuron's piece** (converted to kelvin). Their sum, plus the output bias, is the thick curve. **The network builds a curve out of bent pieces.**

In [ ]:
x_su = su_using(n_smooth, train_x)
total = np.zeros(len(n_smooth)) + b_out

plt.figure(figsize=(7, 4.5))
for j in np.arange(4):
    piece = w_out.item(j) * np.tanh(w_in.item(j) * x_su + b_in.item(j))
    total = total + piece
    plt.plot(n_smooth, piece * np.std(train_y), lw=1.5, label="neuron " + str(j + 1))
plt.plot(n_smooth, total * np.std(train_y), "k", lw=3, label="sum of pieces + bias")
plt.axhline(0, color="gray", lw=0.5)
plt.xlabel("number of carbons"); plt.ylabel("contribution [K, relative to mean bp]")
plt.legend(fontsize=9)
plt.show()

### 2.4 Weight heatmap

For bigger networks a table of weights gets hard to read, so we color each weight instead: **red = positive, blue = negative, white = near zero, darker = bigger**.

#### Define the heatmap function
Run this helper cell. You don't need to understand every plotting line.

In [ ]:
def plot_weight_heatmaps(model, input_names, title=""):
    "One heatmap per layer of weights. Red = positive, blue = negative."
    n_layers = len(model.coefs_)
    widths = []
    for W in model.coefs_:
        widths.append(max(W.shape[1], 2))
    fig, axes = plt.subplots(1, n_layers, figsize=(3 + 0.8 * sum(widths), 1.2 + 0.55 * max(model.coefs_[0].shape[0], 4)),
                             gridspec_kw={"width_ratios": widths})
    biggest = 0
    for W in model.coefs_:
        biggest = max(biggest, np.max(np.abs(W)))
    for k in np.arange(n_layers):
        W = model.coefs_[k]
        ax = axes[k]
        image = ax.imshow(W, cmap="RdBu_r", vmin=-biggest, vmax=biggest, aspect="auto")
        ax.grid(False)
        if k == 0:
            rows = list(input_names)
        else:
            rows = []
            for i in np.arange(W.shape[0]):
                rows.append("L" + str(k) + " n" + str(i + 1))
        cols = []
        for j in np.arange(W.shape[1]):
            cols.append("bp" if k == n_layers - 1 else "n" + str(j + 1))
        ax.set_yticks(np.arange(W.shape[0])); ax.set_yticklabels(rows)
        ax.set_xticks(np.arange(W.shape[1])); ax.set_xticklabels(cols)
        ax.set_title("Output layer" if k == n_layers - 1 else "Into hidden layer " + str(k + 1))
        for i in np.arange(W.shape[0]):
            for j in np.arange(W.shape[1]):
                value = W[i, j]
                label = str(np.round(abs(value), 1)) if abs(value) < 0.05 else str(np.round(value, 1))
                ax.text(j, i, label, ha="center", va="center", fontsize=9,
                        color="white" if abs(value) > 0.6 * biggest else "black")
    fig.colorbar(image, ax=axes, shrink=0.8, label="weight")
    fig.suptitle(title, y=1.06)
    plt.show()

#### Heatmap of the alkane network

In [ ]:
plot_weight_heatmaps(nn_alk, ["carbons"], "Alkane network: 4 tanh neurons")

### 2.5 Network diagram

Same weights, drawn as a network. Line **thickness** = size of the weight, **color** = sign (red +, blue −).

#### Define the network-diagram function
Run this helper cell; we'll use it again for the alcohols.

In [ ]:
def plot_network(model, input_names, title=""):
    "Draw the network: line thickness = size of weight, red = positive, blue = negative."
    sizes = [model.coefs_[0].shape[0]]
    for W in model.coefs_:
        sizes.append(W.shape[1])
    tallest = max(sizes)
    positions = []                       # (x, y) of every node, layer by layer
    for layer in np.arange(len(sizes)):
        n = sizes[layer]
        ys = np.linspace(0, tallest - 1, n) if n > 1 else np.array([(tallest - 1) / 2])
        layer_positions = []
        for y in ys:
            layer_positions.append((layer, tallest - 1 - y))
        positions.append(layer_positions)
    biggest = 0
    for W in model.coefs_:
        biggest = max(biggest, np.max(np.abs(W)))
    plt.figure(figsize=(2.6 * len(sizes), 0.45 * tallest + 2))
    for k in np.arange(len(model.coefs_)):
        W = model.coefs_[k]
        for i in np.arange(W.shape[0]):
            for j in np.arange(W.shape[1]):
                x1, y1 = positions[k][i]
                x2, y2 = positions[k + 1][j]
                color = "#d03b3b" if W[i, j] > 0 else "#2a78d6"
                plt.plot([x1, x2], [y1, y2], color=color, lw=4 * abs(W[i, j]) / biggest, alpha=0.7)
    for layer in np.arange(len(positions)):
        for (x, y) in positions[layer]:
            plt.scatter(x, y, s=250, color="white", edgecolor="black", zorder=3)
    for i in np.arange(len(input_names)):
        x, y = positions[0][i]
        plt.text(x - 0.12, y, input_names[i], ha="right", va="center")
    x, y = positions[-1][0]
    plt.text(x + 0.12, y, "bp", ha="left", va="center")
    plt.axis("off")
    plt.title(title)
    plt.show()

#### Diagram of the alkane network

In [ ]:
plot_network(nn_alk, ["carbons"], "Alkane network: 4 tanh neurons")

### 2.6 ✏️ Your turn: change the activation and the number of neurons

#### Define a function that builds, trains, scores, and plots in one step
It repeats everything from 2.2 in one call.

In [ ]:
def try_alkane_network(neurons, activation, seed=0):
    "Build, train, and plot an alkane network with one hidden layer."
    if activation not in make_array("identity", "logistic", "tanh", "relu"):
        raise ValueError('activation must be "identity", "logistic", "tanh", or "relu" (with quotes)')
    if type(neurons) != int or neurons < 1:
        raise ValueError("neurons must be a whole number, like 4 or 16")
    # create and train
    model = MLPRegressor(hidden_layer_sizes=(neurons,), activation=activation,
                         solver="lbfgs", alpha=0.001, max_iter=20000, random_state=seed)
    model.fit(X_alk, y_alk)
    # score
    test_rmse = rmse(test_alk.column("bp"), predict_alkane_bp(model, test_alk.column("carbons")))
    # plot
    plt.figure(figsize=(6, 3.8))
    plt.scatter(alkanes.column("carbons"), alkanes.column("bp"), color="gray", label="data")
    plt.plot(n_smooth, alk_slope * n_smooth + alk_intercept, "k--", label="regression line")
    plt.plot(n_smooth, predict_alkane_bp(model, n_smooth), lw=3, label=activation + ", " + str(neurons) + " neurons")
    plt.title(f"Test RMSE: {test_rmse:.1f} K   (line: {alk_line_rmse:.1f} K)")
    plt.xlabel("number of carbons"); plt.ylabel("bp [K]"); plt.legend()
    plt.show()
    return model

#### ✏️ Try your combinations
Try **at least two**, for example `"identity"` with 4 neurons (predict first: what shape will it draw?), `"relu"` with 4 neurons, and `"relu"` with 16 neurons.

> 💡 If `relu` draws a straight line or something strange, some neurons have **"died"**: their weighted sum is negative for every molecule, so relu outputs 0 and they stop learning. More neurons or a different `seed` usually fixes it.

In [ ]:
# ✏️ Fill in a number of neurons and an activation ("identity", "logistic", "tanh", "relu")
my_alkane_model = try_alkane_network(..., ...)

In [ ]:
# ✏️ Try another combination
my_alkane_model_2 = try_alkane_network(..., ...)

### 2.7 ⚠️ Beyond the data: predicting triacontane (C₃₀H₆₂)
Our data stops at 20 carbons. Triacontane's real boiling point is about **723 K** (450 °C). What do the models predict? Predicting outside the range of the training data is called **extrapolation**.

In [ ]:
c30 = make_array(30)
print("Actual triacontane bp:   723 K")
print(f"Regression line:         {(alk_slope * c30 + alk_intercept).item(0):.0f} K")
print(f"Neural network (4 tanh): {predict_alkane_bp(nn_alk, c30).item(0):.0f} K")

The neural network is much closer, but notice it is still off by tens of kelvin. **No model has seen what happens beyond its data.** A tanh network eventually flattens out, because each tanh piece levels off. Both models are guessing out there; the curved one just guesses better here.

## 3. Back to the Lab 10 Alcohols

Now the harder, real-world case: the 72 alcohols.

#### Load the data and split it exactly like Lab 10
Same seed (1013) and 80% training, so the numbers match your Lab 10 work.

In [ ]:
ROH_data = Table.read_table("ROH_data.csv")
np.random.seed(1013)
train_roh, test_roh = ROH_data.split(int(0.8 * ROH_data.num_rows))
print(train_roh.num_rows, "training and", test_roh.num_rows, "test alcohols")

#### Fit the regression line with your toolbox and score it
This should match your Lab 10 Question 2.4.

In [ ]:
roh_slope = slope(train_roh, "MW", "bp")
roh_intercept = intercept(train_roh, "MW", "bp")

line_rmse = rmse(test_roh.column("bp"), roh_slope * test_roh.column("MW") + roh_intercept)
line_train_rmse = rmse(train_roh.column("bp"), roh_slope * train_roh.column("MW") + roh_intercept)
print(f"Regression line (MW) test RMSE: {line_rmse:.1f} K")

### 3.1 Wrap it all in one function
With several features, `X` becomes a 2-D array with **one column per feature**. **Read these functions.** Every step is one you did in Part 2.

#### Define a function to turn feature columns into `X`

In [ ]:
def table_to_X(tbl, features):
    "Stack feature columns side by side: one row per molecule, one column per feature."
    columns = []
    for name in features:
        columns.append(tbl.column(name))
    return np.column_stack(columns)

table_to_X(train_roh, make_array("MW", "cyclic"))[:3]      # first 3 rows, as an example

#### Define a function for the neural network
It prepares the data, creates and trains the network, and returns the model with its training and test RMSE.

In [ ]:
def fit_alcohol_network(features, layers, activation, seed=0):
    "Train a network on the alcohol training set. Returns (model, train RMSE, test RMSE)."
    if type(layers) != tuple:
        raise ValueError("layers must be a tuple like (8,) or (8, 4). Don't forget the comma in (8,)!")
    if activation not in make_array("identity", "logistic", "tanh", "relu"):
        raise ValueError('activation must be "identity", "logistic", "tanh", or "relu"')
    # 1. prepare the data in standard units (training mean and SD)
    X_train = table_to_X(train_roh, features)
    X_test = table_to_X(test_roh, features)
    y_train = train_roh.column("bp")
    means, sds = np.mean(X_train, axis=0), np.std(X_train, axis=0)   # axis=0: down each column
    # 2. create and train
    model = MLPRegressor(hidden_layer_sizes=layers, activation=activation,
                         solver="lbfgs", alpha=0.1, max_iter=5000, random_state=seed)
    model.fit((X_train - means) / sds, su_using(y_train, y_train))
    # 3. predict in kelvin and score
    to_kelvin = lambda X: model.predict((X - means) / sds) * np.std(y_train) + np.mean(y_train)
    train_rmse = rmse(y_train, to_kelvin(X_train))
    test_rmse = rmse(test_roh.column("bp"), to_kelvin(X_test))
    return model, train_rmse, test_rmse

#### Make a results Table to compare models
`record` adds one row each time we try a model. It starts with the regression line.

In [ ]:
results = Table().with_columns("Model", make_array(), "Features", make_array(), "Activation", make_array(),
                               "Layers", make_array(), "Train RMSE (K)", make_array(), "Test RMSE (K)", make_array())
results = results.with_row(["Regression line", "MW", "-", "-", np.round(line_train_rmse, 1), np.round(line_rmse, 1)])

def record(name, features, activation, layers, train_rmse, test_rmse):
    "Add one model's results as a new row of the results Table."
    global results
    results = results.with_row([name, ", ".join(features), activation, str(layers),
                                np.round(train_rmse, 1), np.round(test_rmse, 1)])
results

### 3.2 Worked example: a neural network with MW only
Same single feature as the line, but now the model can bend.

#### Train and record

In [ ]:
features_a = make_array("MW")
model_a, train_a, test_a = fit_alcohol_network(features_a, (8,), "tanh")
record("NN, MW only", features_a, "tanh", (8,), train_a, test_a)

#### Compare with the line

In [ ]:
print(f"Regression line test RMSE:     {line_rmse:.1f} K")
print(f"Neural network (MW) test RMSE: {test_a:.1f} K")

**No real improvement!** Why? Remember Lab 10 Question 2.4 Discussion: molecules with the **same MW** can have **different** boiling points. Look at the column of points at MW ≈ 102 below: same MW, boiling points spread over 35 K. No curve drawn through MW alone, however bendy, can hit all of them. The problem isn't the *shape*; it's **missing information**.

#### Plot: same MW, different boiling points

In [ ]:
ROH_data.scatter("MW", "bp", group="degree", s=40)
plt.title("Same MW, different bp: degree (1, 2, 3) explains part of the spread")
plt.show()

### 3.3 Worked example: give the network more information
Let's add `degree` (primary/secondary/tertiary) and `cyclic` (ring or not).

#### Train and record

In [ ]:
features_b = make_array("MW", "degree", "cyclic")
model_b, train_b, test_b = fit_alcohol_network(features_b, (8,), "tanh")
record("NN, 3 features", features_b, "tanh", (8,), train_b, test_b)
print(f"Neural network (MW, degree, cyclic) test RMSE: {test_b:.1f} K")

#### Heatmap

In [ ]:
plot_weight_heatmaps(model_b, features_b, "Alcohol network: 3 features, 8 tanh neurons")

**Reading the heatmap:** each column is one hidden neuron; read down to see which molecular features it responds to. A row that's red or blue in many columns is a feature the network uses a lot. In the output panel, red neurons push the predicted bp **up** and blue ones push it **down**.

**Shapes:** `coefs_[0]` has one row per input feature and one column per hidden neuron, here 3 × 8. `coefs_[1]` is 8 × 1 (8 neurons → 1 boiling point).

#### Network diagram

In [ ]:
plot_network(model_b, features_b, "Alcohol network: 3 features, 8 tanh neurons")

### 3.4 ✏️ Your turn: build your own alcohol model
Choose any **features** from `"MW"`, `"carbons"`, `"degree"`, `"aliphatic"`, `"avg_aryl_position"`, `"cyclic"`, any **layers** (try at least one model with **two** hidden layers, like `(8, 4)`), and any **activation**. Re-run the three cells below with different choices; every run is added to the results table.

💡 Include one model with `"identity"` and several features. That's a network that can't bend, which makes it **multiple linear regression**. Is it better or worse than the bendy networks?

#### ✏️ Choose your model

In [ ]:
my_features = make_array(...)      # e.g. make_array("MW", "degree", "cyclic", "aliphatic")
my_layers = ...                    # e.g. (8,) or (8, 4)
my_activation = ...                # "identity", "logistic", "tanh", or "relu"

#### Train, record, and score

In [ ]:
my_model, my_train, my_test = fit_alcohol_network(my_features, my_layers, my_activation)
record("My model", my_features, my_activation, my_layers, my_train, my_test)
print(f"Train RMSE: {my_train:.1f} K    Test RMSE: {my_test:.1f} K")

#### Heatmap and network diagram of your model

In [ ]:
plot_weight_heatmaps(my_model, my_features, my_activation + " " + str(my_layers))
plot_network(my_model, my_features, my_activation + " " + str(my_layers))

### 3.5 Compare all your models

#### Results table, best first

In [ ]:
results.sort("Test RMSE (K)").show()

#### Bar chart of training and test error

In [ ]:
labels = make_array()
for i in np.arange(results.num_rows):
    labels = np.append(labels, str(i + 1) + ". " + results.column("Model").item(i))
results.with_column("Model", labels).barh("Model", make_array("Train RMSE (K)", "Test RMSE (K)"))
plt.title("Lower is better")
plt.show()

**Things to look for** (you'll write about these next):
- **Overfitting:** a train RMSE much *smaller* than the test RMSE means the model memorized the training molecules instead of learning the pattern.
- **More information vs. more flexibility:** which helped more: adding features, or adding neurons and layers?
- With only 57 training molecules, a simpler model can win. That's a real, useful finding.

## 4. ✏️ Reflection
Double-click each answer cell and write **2–4 sentences**. Use your numbers and plots as evidence.

<font color=blue>**4.1**</font> In your own words, what does a single neuron compute? What is the job of the activation function?

*Your answer:*

<font color=blue>**4.2**</font> Why could the neural network follow the alkane curve when the regression line could not? Use the "pieces" plot in your explanation.

*Your answer:*

<font color=blue>**4.3**</font> For the alcohols, a network using MW alone didn't beat the line, but adding features helped. What does that tell you about when a neural network is (and isn't) the right tool?

*Your answer:*

<font color=blue>**4.4**</font> What did the weight heatmap of your own model show? Describe one specific thing you noticed, and whether your best model overfit.

*Your answer:*

## 5. 📝 Quick Check: Neural Networks
Click an answer to get instant feedback. Keep trying until you get it right.

In [ ]:
with open("questions10_nn.json", "r") as f:
    display_quiz(json.load(f), shuffle_answers=True)

<font color=blue>**5.1**</font> How many questions did you answer correctly on the **first try** (out of 8)? For any you missed, explain the correct idea in your own words.

*Your answer:*

## 6. 🏁 Submitting This Extension

**If your instructor offers extra credit for this extension, be sure to follow these steps:**

1. ✅ **Check your work:** every `...` is replaced, the reflection (4.1–4.4) and quiz summary (5.1) are answered, and **Kernel → Restart Kernel and Run All Cells** runs with no red errors.
2. 💾 **Save** the notebook (Ctrl/Cmd + S).
3. 📤 **Export an HTML copy** ([instructions](https://laserchemist.github.io/data/submission-instructions.html#html)) and upload that HTML file to the **Lab 10 Extension** assignment on Canvas.